In [1]:
from glob import glob
import re
import nltk

nltk_resources = [
    'tokenizers/punkt', 
    'averaged_perceptron_tagger_eng',
    'corpora/stopwords', 
    'help/tagsets'
]

for resource in nltk_resources:
    try:
        nltk.data.find(resource)
    except LookupError:
        nltk.download(resource)

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\student\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [2]:
import pandas as pd 
import numpy as np 
import glob
from lxml import etree

# Change this to where your files are
source_dir = "sources"
source_files_paths = glob.glob(f"{source_dir}/*.xml")

# Iterate through files, grab XML info, and save to a dictionary
docs = []
for i, source_file_path in enumerate(source_files_paths):

    tree = etree.parse(source_file_path)
    root = tree.getroot()
    
    pub_id_str = root.xpath("//front//article-meta//article-id")[1].text

    title_el = root.xpath("//front//article-title")[0]
    title_str = " ".join([t.strip() for t in title_el.itertext()])

    year_str = root.xpath("//front//pub-date/year")[0].text
    month_str = root.xpath("//front//pub-date/month")[0].text
    day_str = root.xpath("//front//pub-date/day")[0].text
    date_str = "-".join([year_str, month_str, day_str])

    kwd_els = root.xpath("//kwd")
    kwd_strs = [kwd_el.text for kwd_el in kwd_els]

    p_els = root.xpath("//body//p")
    p_strs = []
    for p_el in p_els:
        etree.strip_elements(p_el, "xref", with_tail=False)
        p_str = etree.tostring(p_el, method="text", encoding="unicode")
        p_strs.append(p_str)
    
    docs.append({
        'doc_id': pub_id_str,
        'doc_title': title_str,
        'doc_date': date_str,
        'doc_content': " ".join(p_strs),
        'doc_kws': kwd_strs,
        'doc_file_path': source_file_path
    })

# Convert dictionary to data frame
DOC = pd.DataFrame(docs) 
del(docs)

In [3]:
DOC

""


In [5]:
# Parse into Sentences with NLTK
SENTS = DOC.doc_content.apply(lambda x: pd.Series(nltk.sent_tokenize(x), dtype='string'))\
    .stack().to_frame('sent_str')
SENTS.index.names = ['doc_id','sent_num']

AttributeError: 'DataFrame' object has no attribute 'doc_content'

In [6]:
# Parse into Tokens with NLTK
TOKENS = SENTS.sent_str.apply(lambda x: pd.Series(nltk.pos_tag(nltk.word_tokenize(x))))\
    .stack().to_frame('pos_tuple')
TOKENS['pos'] = TOKENS.pos_tuple.apply(lambda x: x[1])
TOKENS['pos_group'] = TOKENS.pos.str[:2]
TOKENS['token_str'] = TOKENS.pos_tuple.apply(lambda x: x[0])
TOKENS = TOKENS.drop('pos_tuple', axis=1)
TOKENS['term_str'] = TOKENS.token_str.str.lower().str.replace(r"[\W_]+", "", regex=True) 
TOKENS = TOKENS[TOKENS.term_str != ''].copy()
TOKENS.index.names = SENTS.index.names + ['token_num']

NameError: name 'SENTS' is not defined

In [ ]:
TOKENS

pos pos_group     token_str      term_str
doc_id sent_num token_num                                            
0      0        0           NNP        NN    Scientific    scientific
                1            NN        NN        impact        impact
                2           NNS        NN      measures      measures
                3           VBP        VB           are           are
                4            RB        RB  increasingly  increasingly
...                         ...       ...           ...           ...
8      751      20         PRP$        PR           its           its
                21           JJ        JJ     continued     continued
                22           NN        NN      progress      progress
                23           CC        CC           and           and
                24           NN        NN       success       success

[58640 rows x 4 columns]

In [ ]:
# TOKEN = DOC.doc_content.str.split(expand=True).stack().to_frame("token_str")
# TOKEN.index.names = ['doc_id','token_num']
# TOKEN['term_str'] = TOKEN.token_str.str.lower().str.replace(r"[^a-z]+", "", regex=True)
# TOKEN = TOKEN[~(TOKEN.term_str == "")]
# TOKEN

In [ ]:
DOCKW = DOC.doc_kws.apply(pd.Series).stack().to_frame('doc_kw')
DOCKW.index.names = ['doc_id', 'kw_num']
DOCKW.head()

doc_kw
doc_id kw_num                    
0      0             Soil science
       1            Bibliometrics
       3            Impact factor
       4                Citations
       5       Transfer functions

In [ ]:
KW = DOCKW.doc_kw.value_counts().to_frame('n')
KW.head()

,n
doc_kw,
AI,2
Soil science,1
Sustainability,1
Protein quality,1
Net energy,1
